# MADRL · Entrenamiento modular en Google Colab

Cliente de la CLI de la campaña nueva (`python -m uc3m.application.training.cli`): no
contiene otra lógica de entrenamiento. Reutiliza la instalación Python 3.9 del tutorial
original sin sobrescribir sus celdas. Cada celda de código empieza con `# [Cn]` (C1–C11).

Cada backend entrena con su capacidad nativa, sin recortes silenciosos. Antes de entrenar,
la entrada nativa mide RAM y VRAM y rechaza (`insufficient_resources:<algoritmo>:<ram|vram>`)
si no alcanzan los requisitos de `configs/algorithms/native.yaml` (episodios de 8760 pasos):
HAPPO 8 GiB RAM / 4 GiB VRAM; MASAC 50 GiB RAM / 74 GiB VRAM; MATD3 40 GiB RAM / 4 GiB VRAM; MAAC 8 GiB RAM / 4 GiB VRAM.

**MASAC con los valores del autor (lote del crítico de 32 episodios) es el que fija la GPU:**
necesita 74 GiB de VRAM con `rnn_hidden_dim` 128, su único candidato (D-M3-8). Una A100/H100
de 80 GB expone unas 74,5 GiB, así que cabe con poco margen. Un rechazo
`insufficient_resources:MASAC:vram` deja incompleta la campaña: es un fallo que se revisa, no un
resultado aceptable. Los requisitos son estimaciones. C6 guarda el pico de VRAM de cada etapa, y el piloto
no llega al lote de 32 episodios, de modo que su pico no es el máximo de la campaña.
Usar Colab Pro con RAM alta; la disponibilidad de GPU y la duración de la sesión dependen
de Colab.

**Importante:** los cambios locales aún no publicados NO aparecen al clonar GitHub. Subir
el paquete `colab_modular_overlay.zip` generado junto a este cuaderno; contiene código y
dataset actuales con su hash por archivo, sin entornos ni credenciales.


In [ ]:
# [C1] Paquete y checkout exacto
from google.colab import files
from pathlib import Path
import importlib.util, io, json, os, subprocess, zipfile

ORIGIN = 'https://github.com/Mac-Tapia/MADRLCitytleranflexresdr.git'
PROJECT_ROOT = Path('/content/MADRLCitytleranflexresdr')
uploaded = files.upload()
payload = uploaded.get('colab_modular_overlay.zip')
if payload is None:
    raise RuntimeError('Sube colab_modular_overlay.zip generado en tu proyecto local')
bundle = zipfile.ZipFile(io.BytesIO(payload))
BUNDLE_MANIFEST = json.loads(bundle.read('colab_bundle_manifest.json'))
import sys
spec = importlib.util.spec_from_loader('colab_session', loader=None)
cs = importlib.util.module_from_spec(spec)
sys.modules['colab_session'] = cs                   # dataclasses need the module registered
exec(compile(bundle.read('colab_session.py').decode('utf-8'), 'colab_session.py', 'exec'), cs.__dict__)
if not PROJECT_ROOT.exists():
    subprocess.run(['git', 'clone', '--recursive', ORIGIN, str(PROJECT_ROOT)], check=True)
    subprocess.run(['git', 'checkout', BUNDLE_MANIFEST['commit']], cwd=PROJECT_ROOT, check=True)
    subprocess.run(['git', 'submodule', 'update', '--init', '--recursive'], cwd=PROJECT_ROOT, check=True)
applied = cs.apply_overlay(bundle, PROJECT_ROOT)
reasons = cs.verify_checkout(PROJECT_ROOT, BUNDLE_MANIFEST, origin=ORIGIN)
if reasons:
    raise RuntimeError(f'El checkout no coincide con el paquete local: {reasons[:20]}')
os.chdir(PROJECT_ROOT)
print('Paquete aplicado:', applied['written'], 'archivos; retirados del clon:', applied['removed'])
print('Checkout verificado contra el paquete (commit, submódulos, hash por archivo).')


## Instalación original del proyecto
Se reutiliza la celda Python 3.9 del tutorial, con dependencias fijadas a `uv.lock` y al entorno local.


In [ ]:
# [C2] Instalación Python 3.9 fijada
import platform
# 1.3 Instalar dependencias del proyecto de forma reproducible
# Usa Python 3.9 del proyecto. Si Colab entrega kernel 3.11, crea/usa .venv39-citylearn-v3.
import importlib
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

PROJECT_DIR = '/content/MADRLCitytleranflexresdr'
PROJECT_ROOT = Path(PROJECT_DIR)
if not PROJECT_ROOT.exists() and Path.cwd().name == 'MADRLCitytleranflexresdr':
    PROJECT_ROOT = Path.cwd()
    PROJECT_DIR = str(PROJECT_ROOT)

VENV_DIR = PROJECT_ROOT / '.venv39-citylearn-v3'
SETUP_LOG = Path('/tmp/madrl_py39_setup.log')
CONSTRAINTS = Path('/tmp/madrl_compat.txt')
PYTHON_REQUIRED = (3, 9)
PYTHON_MIN = PYTHON_REQUIRED
PYTHON_MAX_EXCLUSIVE = (3, 10)
PYTORCH_INDEX_CU126 = 'https://download.pytorch.org/whl/cu126'
PYTORCH_INDEX_CU128 = 'https://download.pytorch.org/whl/cu128'
TORCH_PACKAGES = ('torch==2.8.0', 'torchvision==0.23.0')

def detect_pytorch_cuda_wheel():
    """Blackwell (sm_120, RTX PRO 6000) requiere wheels cu128; A100/H100 usan cu126."""
    try:
        name = subprocess.check_output(
            ['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
            text=True, stderr=subprocess.DEVNULL,
        ).strip().splitlines()[0].strip()
    except Exception:
        return PYTORCH_INDEX_CU126, None
    upper = name.upper()
    blackwell = any(k in upper for k in ('BLACKWELL', 'RTX PRO 6000', 'RTX 50'))
    index = PYTORCH_INDEX_CU128 if blackwell else PYTORCH_INDEX_CU126
    return index, name

PYTORCH_INDEX_URL, _DETECTED_GPU = detect_pytorch_cuda_wheel()
print(f'[torch] GPU: {_DETECTED_GPU or "(sin nvidia-smi)"} -> index {PYTORCH_INDEX_URL.split("/")[-1]}')

COMPAT_WHEELS = [
    'numpy==1.23.5',
    'pandas==2.0.3',
    'scipy==1.10.1',
    'scikit-learn==1.2.2',
    'matplotlib==3.7.5',
    'seaborn==0.12.2',
]
PINNED = {
    'numpy': '1.23.5',
    'pandas': '2.0.3',
    'scipy': '1.10.1',
    'scikit-learn': '1.2.2',
    'matplotlib': '3.7.5',
    'seaborn': '0.12.2',
    'gymnasium': '0.28.1',
    'pettingzoo': '1.12.0',
}
BASE_DEPS = [
    *COMPAT_WHEELS,
    'pyyaml==6.0.3',
    'requests==2.32.5',
    'tqdm==4.68.2',
    'psutil==7.2.2',
    'platformdirs==4.4.0',
    'protobuf==3.20.3',
    'gymnasium==0.28.1',
    'pettingzoo==1.12.0',
    'gym==0.20.0',
    'tensorboardX==2.6.5',
    'setproctitle==1.3.7',
    'simplejson==4.1.1',
    'absl-py==2.3.1',
    'dm-tree==0.1.8',
    'importlib-metadata==8.7.1',
]
NO_DEPS_UTILS = [
    'supersuit==3.2.0',
    'icecream==2.1.3',
]
EDITABLES = [
    'CityLearn/',
    'external/HARL/',
    'external/off-policy/',
    'external/MAAC/',
    'external/MARL/src/',
]
BINARY_DEPS = ('numpy', 'pandas', 'scipy', 'scikit-learn', 'matplotlib', 'seaborn')

ABI_CHECK = """
import importlib
import json
import sys

modules = {
    'torch': 'torch',
    'numpy': 'numpy',
    'pandas': 'pandas',
    'scipy': 'scipy',
    'scikit-learn': 'sklearn',
    'matplotlib': 'matplotlib',
    'seaborn': 'seaborn',
    'gym': 'gym',
    'gymnasium': 'gymnasium',
    'pettingzoo': 'pettingzoo',
    'citylearn.v3.environment': 'citylearn.v3.environment',
}
CRITICAL = {'numpy', 'scipy', 'sklearn', 'gymnasium', 'pettingzoo', 'citylearn.v3.environment'}
versions = {'python': sys.version.split()[0], 'executable': sys.executable}
failures = {}
for label, module_name in modules.items():
    try:
        module = importlib.import_module(module_name)
        versions[label] = getattr(module, '__version__', 'importado')
    except Exception as exc:
        failures[label] = repr(exc)
        versions[label] = f'ERROR: {exc}'
try:
    import torch
    versions['torch_cuda_available'] = bool(torch.cuda.is_available())
    versions['torch_cuda'] = getattr(torch.version, 'cuda', None)
except Exception as exc:
    versions['torch_cuda_error'] = repr(exc)
print(json.dumps(versions, indent=2, sort_keys=True))
critical_failures = {k: v for k, v in failures.items() if k in CRITICAL}
if critical_failures:
    print('CRITICAL_FAILURES: ' + json.dumps(critical_failures), file=sys.stderr)
    sys.exit(1)
elif failures:
    print('NON_CRITICAL_FAILURES: ' + json.dumps(failures), file=sys.stderr)
"""


def write_log(text):
    SETUP_LOG.parent.mkdir(parents=True, exist_ok=True)
    with SETUP_LOG.open('a', encoding='utf-8') as f:
        f.write(text)
        if not text.endswith('\n'):
            f.write('\n')


def print_log_tail(lines=80):
    if not SETUP_LOG.exists():
        return
    tail = SETUP_LOG.read_text(encoding='utf-8', errors='replace').splitlines()[-lines:]
    print(f'\n[TAIL {SETUP_LOG}]')
    print('\n'.join(tail))


def run(cmd, *, cwd=None, env=None, check: bool = True):
    cmd = [str(part) for part in cmd]
    message = '+ ' + ' '.join(cmd)
    print(message)
    write_log('\n' + message)
    proc = subprocess.run(
        cmd,
        cwd=str(cwd or PROJECT_ROOT),
        env=env,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    if proc.stdout:
        write_log(proc.stdout)
    if check and proc.returncode != 0:
        print_log_tail()
        raise RuntimeError(
            f'Comando fallo con exit={proc.returncode}: {message}. '
            f'Log completo: {SETUP_LOG}'
        )
    return proc


def run_shell(script, *, cwd=None, env=None, check: bool = True):
    return run(['bash', '-lc', script], cwd=cwd, env=env, check=check)


def venv_python_path():
    if os.name == 'nt':
        return VENV_DIR / 'Scripts' / 'python.exe'
    return VENV_DIR / 'bin' / 'python'


def python_info(python):
    python = str(python)
    if not Path(python).exists() and python != sys.executable:
        return None
    code = """
import json
import sys
print(json.dumps({
    'executable': sys.executable,
    'version': sys.version.split()[0],
    'version_info': list(sys.version_info[:3]),
}))
"""
    result = subprocess.run([python, '-c', code], capture_output=True, text=True)
    if result.returncode != 0:
        return None
    return json.loads(result.stdout)


def same_executable(left, right):
    try:
        return Path(left).resolve() == Path(right).resolve()
    except Exception:
        return str(left) == str(right)


def setup_env():
    env = os.environ.copy()
    home_bin = str(Path.home() / '.local' / 'bin')
    env['PATH'] = home_bin + os.pathsep + env.get('PATH', '')
    return env


def ensure_uv(env):
    uv = shutil.which('uv', path=env.get('PATH'))
    if uv:
        return uv
    run_shell('curl -LsSf https://astral.sh/uv/0.12.17/install.sh | sh', env=env)
    uv = shutil.which('uv', path=env.get('PATH'))
    if uv:
        return uv
    candidate = Path.home() / '.local' / 'bin' / 'uv'
    if candidate.exists():
        return str(candidate)
    raise RuntimeError('uv no quedo disponible en PATH despues de instalarlo.')


def ensure_project_python39():
    current_info = python_info(sys.executable)
    if current_info and tuple(current_info['version_info'][:2]) == PYTHON_REQUIRED:
        return sys.executable

    project_python = venv_python_path()
    project_info = python_info(project_python)
    if project_info and tuple(project_info['version_info'][:2]) == PYTHON_REQUIRED:
        return str(project_python)

    if platform.system() == 'Windows':
        raise RuntimeError(
            'El kernel actual no es Python 3.9. En Windows selecciona '
            '.venv39-citylearn-v3 como kernel o recrea el entorno segun docs/architecture/local.md.'
        )

    env = setup_env()
    uv = ensure_uv(env)
    print(
        f'Kernel actual: Python {sys.version.split()[0]} ({sys.executable}). '
        f'Creando entorno de proyecto Python 3.9 en {VENV_DIR}.'
    )
    run([uv, 'python', 'install', '3.9'], cwd=PROJECT_ROOT, env=env)
    run([uv, 'venv', '--python', '3.9', str(VENV_DIR)], cwd=PROJECT_ROOT, env=env)

    project_info = python_info(project_python)
    if not project_info or tuple(project_info['version_info'][:2]) != PYTHON_REQUIRED:
        raise RuntimeError(f'No se pudo crear un Python 3.9 valido en {project_python}')
    return str(project_python)


def pip_install(*args):
    cmd = [PROJECT_PYTHON, '-m', 'pip', 'install', '--disable-pip-version-check', *args]
    run(cmd)


def installed_version(package):
    code = """
import importlib.metadata as im
import sys
package = sys.argv[1]
names = (package, package.replace('-', '_'), package.replace('_', '-'))
for name in dict.fromkeys(names):
    try:
        print(im.version(name))
        raise SystemExit(0)
    except im.PackageNotFoundError:
        pass
raise SystemExit(1)
"""
    result = subprocess.run([PROJECT_PYTHON, '-c', code, package], capture_output=True, text=True)
    return result.stdout.strip() if result.returncode == 0 else None


def torch_gpu_ready():
    """True solo si PyTorch puede ejecutar un kernel real en la GPU (detecta sm_120 sin cu128)."""
    code = """
import json, sys
try:
    import torch
    info = {
        'version': torch.__version__,
        'cuda_runtime': getattr(torch.version, 'cuda', None),
        'cuda_available': bool(torch.cuda.is_available()),
    }
    if not info['cuda_available']:
        print(json.dumps({**info, 'ready': False, 'reason': 'cuda_unavailable'}))
        raise SystemExit(0)
    cap = torch.cuda.get_device_capability(0)
    info['device'] = torch.cuda.get_device_name(0)
    info['capability'] = list(cap)
    upper = info['device'].upper()
    needs_cu128 = cap[0] >= 12 or any(k in upper for k in ('BLACKWELL', 'RTX PRO 6000', 'RTX 50'))
    info['needs_cu128'] = needs_cu128
    cuda_rt = str(info['cuda_runtime'] or '')
    if needs_cu128 and not cuda_rt.startswith('12.8'):
        print(json.dumps({**info, 'ready': False, 'reason': 'blackwell_needs_cu128'}))
        raise SystemExit(0)
    x = torch.zeros(1, device='cuda')
    _ = (x + 1).item()
    torch.cuda.synchronize()
    print(json.dumps({**info, 'ready': True, 'kernel_ok': True}))
except Exception as exc:
    print(json.dumps({'ready': False, 'reason': 'kernel_failed', 'error': repr(exc)}))
"""
    result = subprocess.run([PROJECT_PYTHON, '-c', code], capture_output=True, text=True)
    line = (result.stdout or '').strip().splitlines()
    if not line:
        return False, {}
    try:
        data = json.loads(line[-1])
    except Exception:
        return False, {}
    if result.stdout.strip():
        print('[torch]', line[-1])
    return bool(data.get('ready')), data


def torch_cuda_available():
    ready, _ = torch_gpu_ready()
    return ready


def restart_runtime(reason):
    print(f'\n[RESTART REQUERIDO] {reason}')
    print('Reinicia el runtime y vuelve a ejecutar desde la celda C1 de este cuaderno.')
    try:
        from IPython.core.getipython import get_ipython
        import time

        print('Colab detectado: reiniciando kernel automaticamente...')
        _ip = get_ipython()
        _kernel = getattr(_ip, 'kernel', None) if _ip is not None else None
        if _kernel is not None:
            _kernel.do_shutdown(restart=True)
        time.sleep(10)
    except Exception:
        pass
    raise RuntimeError(reason)


def repair_binary_abi():
    print('[ABI] Reinstalando ruedas binarias con versiones fijadas (numpy 1.23.5)...')
    pip_install('-q', '--force-reinstall', '--no-cache-dir', *COMPAT_WHEELS)


def verify_subprocess_imports():
    result = subprocess.run([PROJECT_PYTHON, '-c', ABI_CHECK], capture_output=True, text=True)
    if result.stdout.strip():
        print(result.stdout, end='')
    if result.returncode != 0:
        stderr = result.stderr.strip()
        abi_mismatch = any(
            token in stderr
            for token in ('numpy.dtype size changed', 'binary incompatibility', 'numpy.core', 'numpy.strings')
        )
        if abi_mismatch:
            repair_binary_abi()
            result = subprocess.run([PROJECT_PYTHON, '-c', ABI_CHECK], capture_output=True, text=True)
            if result.stdout.strip():
                print(result.stdout, end='')
        if result.returncode != 0:
            if result.stderr.strip():
                print('[STDERR verificacion ABI:]')
                print(result.stderr[-4000:], end='')
            print_log_tail()
            raise RuntimeError('ABI fallo: ' + result.stderr.strip()[-2000:])
    elif result.stderr.strip():
        print('[advertencias ABI (no criticas):]')
        print(result.stderr.strip())


def verify_current_kernel_imports_if_needed():
    if not same_executable(PROJECT_PYTHON, sys.executable):
        print(
            f'Kernel notebook: Python {sys.version.split()[0]} ({sys.executable}). '
            f'Entrenamiento: {PROJECT_PYTHON}. No se importan paquetes del proyecto en el kernel.'
        )
        return

    modules = ('numpy', 'scipy', 'sklearn', 'pandas', 'citylearn.v3.environment')
    failures = {}
    for module_name in modules:
        try:
            importlib.import_module(module_name)
        except Exception as exc:
            failures[module_name] = repr(exc)
    if failures:
        restart_runtime(
            'El kernel actual tiene imports binarios inconsistentes: '
            f'{failures}. Esto ocurre si pip cambio numpy/scipy sin reiniciar.'
        )


if not PROJECT_ROOT.exists():
    raise FileNotFoundError(f'PROJECT_DIR no existe: {PROJECT_ROOT}. Ejecuta primero la celda C1 de este cuaderno.')

SETUP_LOG.write_text('', encoding='utf-8')
PROJECT_PYTHON = ensure_project_python39()
PYTHON = PROJECT_PYTHON
project_info = python_info(PROJECT_PYTHON)
if not project_info or tuple(project_info['version_info'][:2]) != PYTHON_REQUIRED:
    raise RuntimeError(f'Python de proyecto invalido: {project_info}')

os.chdir(PROJECT_DIR)
CONSTRAINTS.write_text('\n'.join(f'{p}=={v}' for p, v in PINNED.items()) + '\n')
print(f"Python proyecto: {project_info['version']} ({PROJECT_PYTHON})")
print(f"Python kernel  : {sys.version.split()[0]} ({sys.executable})")
print(f'Log setup      : {SETUP_LOG}')

# Pip compatible con gym/ray legacy del proyecto.
run([PROJECT_PYTHON, '-m', 'ensurepip', '--upgrade'], check=False)
pip_install('--force-reinstall', 'pip==21.3.1', 'setuptools==65.5.0', 'wheel==0.38.0')

pip_install('-q', *BASE_DEPS)
for package in NO_DEPS_UTILS:
    if installed_version(package.split('==')[0]) is None:
        pip_install('-q', '--no-deps', package)

ready, torch_info = torch_gpu_ready() if installed_version('torch') else (False, {})
if not ready:
    reason = torch_info.get('reason', 'missing_or_incompatible')
    print(f'[torch] Instalando PyTorch ({PYTORCH_INDEX_URL.split("/")[-1]}) — motivo: {reason}')
    pip_install('--force-reinstall', '-q', *TORCH_PACKAGES, '--index-url', PYTORCH_INDEX_URL)
    ready, torch_info = torch_gpu_ready()
if not ready:
    raise RuntimeError(
        'PyTorch no puede ejecutar kernels en esta GPU. '
        f'info={torch_info}. Blackwell (RTX PRO 6000) requiere cu128: '
        'pip install torch torchvision --index-url https://download.pytorch.org/whl/cu128'
    )

for package_dir in EDITABLES:
    pip_install('-q', '--no-deps', '-c', str(CONSTRAINTS), '-e', package_dir)

# Reinstalacion final: evita ruedas pandas/scipy compiladas para numpy 2.x.
pip_install('-q', '--force-reinstall', '--no-cache-dir', *COMPAT_WHEELS)

binary_after = {package: installed_version(package) for package in BINARY_DEPS}
print('Paquetes binarios:', binary_after)

print('\nVerificando ABI en Python 3.9 del proyecto...')
verify_subprocess_imports()
verify_current_kernel_imports_if_needed()
print('\nC2 OK: Python 3.9 del proyecto listo y backends en modo editable.')

In [ ]:
# [C3] Campaña, dataset y Drive por identidad
from google.colab import drive
drive.mount('/content/drive')
import json, yaml
from pathlib import Path

PYTHON = cs.project_python(PROJECT_ROOT, PROJECT_PYTHON)     # el Python 3.9 preparado en C2
CONFIG = 'configs/experiments/iquitos.yaml'
CAMPAIGN = yaml.safe_load((PROJECT_ROOT/CONFIG).read_text())
ALIAS, CAMPAIGN_ID = CAMPAIGN['storage_alias'], CAMPAIGN['campaign_id']
if (ALIAS, CAMPAIGN_ID) != (BUNDLE_MANIFEST['alias'], BUNDLE_MANIFEST['campaign_id']):
    raise RuntimeError('El paquete es de otra campaña que la configuración')
CAMPAIGN_ROOT = PROJECT_ROOT/CAMPAIGN['dataset_root']
ARTIFACTS = PROJECT_ROOT/'outputs/campaigns'/ALIAS
MANIFEST = ARTIFACTS/'campaign_manifest.json'
acceptance = json.loads((CAMPAIGN_ROOT/'dataset_acceptance.json').read_text())
if not acceptance['dataset_verified'] or acceptance['dataset_hash'] != BUNDLE_MANIFEST['dataset_hash']:
    raise RuntimeError('Dataset de la campaña no verificado o distinto al paquete local')
split_manifest = {s['split_id']: s for s in json.loads((CAMPAIGN_ROOT/'split_manifest.json').read_text())['splits']}
for split_id in CAMPAIGN['splits']:
    if split_id not in split_manifest or not (CAMPAIGN_ROOT/'splits'/split_manifest[split_id]['schema_path']).is_file():
        raise RuntimeError(f'Falta el split congelado {split_id}')
snapshot = cs.run_cli([str(PYTHON), '-c', 'import json; from pathlib import Path; '
                       'from uc3m.application.training.preflight import source_snapshot; '
                       "print(json.dumps({'source_sha256': source_snapshot(Path('.'), scope='fresh')}))"],
                      cwd=PROJECT_ROOT, on_line=lambda line: None)
if snapshot.report['source_sha256'] != BUNDLE_MANIFEST['source_sha256']:
    raise RuntimeError('Las fuentes calculadas con el Python del proyecto difieren del paquete')
DRIVE_ROOT = cs.drive_root(Path('/content/drive/MyDrive/MADRL_campaigns'), ALIAS, CAMPAIGN_ID)
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
CLI = [str(PYTHON), '-m', 'uc3m.application.training.cli', '--config', CONFIG]
SYNC = [str(PYTHON), '-m', 'uc3m.application.training.drive_sync', '--artifacts', str(ARTIFACTS),
        '--drive', str(DRIVE_ROOT)]
print('Campaña', CAMPAIGN_ID, '| dataset', acceptance['dataset_hash'][:12], '| Drive', DRIVE_ROOT)


## Restaurar desde Drive los intentos ya terminados

Antes de preparar o lanzar se restauran a `outputs/campaigns/<alias>/` el registro
`treatments.json`, los manifiestos y los intentos terminados que verifican su hash y
`treatment_trust_reasons`, para que la etapa reutilice lo verificado. Un intento
interrumpido es fallido y nunca llegó a Drive (no tenía `completion.json`): su
tratamiento se repite desde cero, sin reanudar. Si hay conflictos o copias rechazadas,
la celda se detiene antes de preparar.


In [ ]:
# [C4] Restaurar intentos verificados desde Drive
restored = cs.run_cli(SYNC + ['--restore', '--campaign-id', CAMPAIGN_ID], cwd=PROJECT_ROOT).report
print('Restaurados completos:', len(restored['restored_complete']), '| fallidos:', len(restored['restored_failed']))
if restored['conflicts'] or restored['rejected']:
    raise RuntimeError(f"Drive y la copia local no coinciden: conflictos {restored['conflicts']}, "
                       f"rechazados {restored['rejected']}")


## Preparar la campaña: no entrena

`prepare` construye el DAG (piloto, HPO, confirmación y sensibilidad) con su presupuesto
de pasos y mide RAM y VRAM para mostrar el plan de esa máquina; no entrena. Un rechazo muestra
el JSON con su razón.


In [ ]:
# [C5] Preparar campaña (no entrena)
prepared = cs.run_cli(CLI + ['--stage', 'prepare'], cwd=PROJECT_ROOT).report
print('Estado:', prepared['status'])
print('Presupuesto:', json.dumps(prepared['budget'], indent=1))
STAGE = 'pilot'  # 'pilot', 'hpo', 'confirm' o 'sensitivity': se muestra su plan aquí y C6 lanza esta misma
for line in cs.phase_plan_lines(prepared, STAGE):        # tal como lo informa prepare, sin lógica propia
    print(line)


## Ejecutar una etapa autorizada

Cada intento empieza en el episodio 0 y nunca se reanuda. Mientras corre, cada intento
se copia a Drive en cuanto escribe su `completion.json`, con hash verificado y sin tocar
el intento en curso; al terminar o si la etapa se interrumpe se hace una última copia.
Un intento interrumpido no tiene `completion.json`: no se copia y, al relanzar tras C4,
su tratamiento se repite desde cero en un intento nuevo; los tratamientos con intento
verificado se reutilizan. Cada etapa necesita la decisión escrita del propietario: es `STAGE`
de C5 y solo se lanza con `EXECUTE_STAGE = True` y la referencia a esa decisión en `OWNER_DECISION`, que
queda en `stages/owner_authorizations.jsonl`. Tras lanzar, vuelve a poner `EXECUTE_STAGE = False`. No se edita `configs/experiments/iquitos.yaml`: es
una entrada de la identidad de la campaña. Si una copia a Drive falla, el entrenamiento sigue y la
celda lo avisa. Códigos de la CLI: 0 completa, 1 incompleta, 2 no autorizada o insumos cambiados.


In [ ]:
# [C6] Ejecutar etapa autorizada con sincronización
EXECUTE_STAGE = False  # True para lanzar la etapa STAGE elegida y revisada en C5
OWNER_DECISION = ''    # dónde y cuándo autorizó el propietario esta etapa, por ejemplo 'traspaso_sesiones.md 2026-12-01 pilot'
RUN_STAGE = None       # se deriva de STAGE; no escribir aquí otra etapa
if EXECUTE_STAGE:
    RUN_STAGE = STAGE
if RUN_STAGE and RUN_STAGE != STAGE:
    raise RuntimeError(f'RUN_STAGE={RUN_STAGE!r} distinto de la etapa revisada en C5 (STAGE={STAGE!r})')
if RUN_STAGE and not OWNER_DECISION.strip():
    raise RuntimeError('Escribe en OWNER_DECISION la decisión del propietario que autoriza esta etapa')
if RUN_STAGE:
    gpu_peak = ARTIFACTS/'stages'/f'{RUN_STAGE}_gpu_memory.json'
    stage = cs.run_stage_with_sync(CLI + ['--stage', RUN_STAGE, '--execute', '--owner-authorized', OWNER_DECISION], SYNC + ['--sync'],
                                   cwd=PROJECT_ROOT, interval_seconds=60, allowed=(0, 1, 2), gpu_peak_path=gpu_peak)
    report = stage.report or {}
    print('returncode', stage.returncode, '| estado', report.get('status'))
    print('Pico de VRAM:', gpu_peak.read_text() if gpu_peak.is_file() else 'sin lectura de nvidia-smi')
    if stage.sync_problems:
        print('ATENCIÓN: hubo', len(stage.sync_problems), 'copias a Drive fallidas; la última:', stage.sync_problems[-1])
    result_path = ARTIFACTS/'stages'/f'{RUN_STAGE}_result.json'
    stage_result = json.loads(result_path.read_text()) if result_path.is_file() else None
    for line in cs.stage_result_lines(report, stage_result):          # tal como lo informa run_stage
        print(line)
    if report.get('status') == 'insufficient_resources':
        stopped = cs.stopped_phase_label(report)                       # report['stopped_at_phase'] por algoritmos
        reasons = [r for phase in report.get('phase_plan') or [] for r in phase.get('reasons') or []]
        raise RuntimeError(f'Etapa detenida en la fase {stopped} por recursos insuficientes '
                           f'(0 trabajadores): {reasons}. La etapa queda incompleta: revisar GPU y RAM de esta sesión.')
    if stage.returncode == 2:
        raise RuntimeError(f"Etapa NO ejecutada (código 2): {report.get('status')} {report.get('reason', '')}")
    if stage.returncode == 1:
        raise RuntimeError('Etapa incompleta (código 1): hay tratamientos sin intento verificado; NO es un éxito')
    if stage.final_sync_failed:
        raise RuntimeError('La etapa terminó, pero la última copia a Drive falló: vuelve a copiar antes de cerrar la sesión')
    print('Etapa completa:', report['status'])
else:
    print(f'Sin entrenamiento: revisa el plan de {STAGE} en C5 y pon EXECUTE_STAGE = True si el propietario autorizó esta etapa.')


## Después del entrenamiento

Las celdas C7–C11 llaman a `python -m uc3m.application.pipeline` con el Python del
proyecto. La última línea es JSON; código 0 = completo o incompleto (informado), 2 =
rechazado con su razón. Nada se sobrescribe: un resultado existente se informa.


In [ ]:
# [C7] Estado por tratamiento
status = cs.run_cli([str(PYTHON), '-m', 'uc3m.application.pipeline', 'status', '--manifest', str(MANIFEST)], cwd=PROJECT_ROOT,
                    on_line=lambda line: None).report
for row in status.get('treatments', status if isinstance(status, list) else []):
    print(row.get('stage'), row.get('treatment_id'), 'aceptado' if row.get('accepted') else 'NO aceptado',
          row.get('reasons'))


## Evaluación congelada

Evalúa con la política congelada cada intento aceptado de la etapa, por defecto en
`validation_2024`. `test_2025` solo se evalúa con un `test_unlock.json` válido. Los
procesos en paralelo son `max(1, cpu_count - 1)`, acotados por la RAM disponible y la
RAM medida de una evaluación (un año de CityLearn con una política, en CPU). Cada evaluación aparece
completa o no aparece, y se copia a Drive mientras la celda corre, de modo que un corte conserva lo terminado.


In [ ]:
# [C9] Evaluación congelada
import os, psutil
RUN_EVALUATION = False
EVAL_STAGE, EVAL_SPLIT = 'hpo', 'validation_2024'   # 'hpo' antes de C8; luego 'confirm' y 'sensitivity'
EVAL_RAM_GIB = 1.45   # medido: 2026-11-01 local Windows CPU: run_frozen_episode(validation_2024, RbcPolicy, scenario E1, seed 1001), 8783 steps in 2744 s; peak working set 1.21 GiB (psutil RSS sampled every 0.5 s: 1.04 GiB) + 0.24 GiB parameters of the largest policy (MATD3, 17 x h=256, ~64.5 M float32, computed)
available = psutil.virtual_memory().available/2**30
workers = max(1, min((os.cpu_count() or 2) - 1, int(available // EVAL_RAM_GIB)))
if RUN_EVALUATION:
    run = cs.run_stage_with_sync([str(PYTHON), '-m', 'uc3m.application.pipeline', 'evaluate', '--manifest', str(MANIFEST), '--stage', EVAL_STAGE,
                                  '--split', EVAL_SPLIT, '--workers', str(workers),
                                  '--project-root', str(PROJECT_ROOT)], SYNC + ['--sync'], cwd=PROJECT_ROOT,
                                 interval_seconds=300, allowed=(0, 2))
    evaluated = run.report or {}
    print('Evaluados:', len(evaluated.get('evaluated', [])), '| existentes:', len(evaluated.get('existing', [])),
          '| fallidos:', evaluated.get('failed'), '| código', run.returncode)
    if run.final_sync_failed:
        raise RuntimeError('La última copia a Drive falló: vuelve a copiar antes de cerrar la sesión')
    if run.returncode != 0:
        raise RuntimeError(f'Evaluación incompleta o rechazada: relanza esta celda; lo evaluado se conserva. {evaluated}')
else:
    print('Sin evaluación; workers previstos:', workers)


## Congelar la configuración de HPO

Va después de C9 porque el congelado exige los intentos de `hpo` evaluados en `validation_2024`.
Orden de la campaña: `hpo` en C6, C9 con `EVAL_STAGE = 'hpo'`, C8, y luego `confirm` y
`sensitivity` en C6, con C9 sobre cada una.


In [ ]:
# [C8] Congelar configuración HPO (validación)
RUN_FREEZE = False
if RUN_FREEZE:
    frozen = cs.run_cli([str(PYTHON), '-m', 'uc3m.application.pipeline', 'freeze', '--manifest', str(MANIFEST), '--config', CONFIG,
                                    '--project-root', str(PROJECT_ROOT)], cwd=PROJECT_ROOT).report
    print(json.dumps(frozen, indent=1)[:4000])
    cs.run_cli(SYNC + ['--sync'], cwd=PROJECT_ROOT, on_line=lambda line: None)


## Análisis, selección y prueba

Orden en `confirm`: C9 con `validation_2024`, C10 con esa misma partición y selección, y solo después C9 y C10
con `test_2025`. La selección nunca se hace mirando la prueba. Un análisis incompleto se rechaza sin publicarse:
se corrige lo que falta y se relanza. En `sensitivity` se analiza cada variante con `ANALYSIS_CANDIDATE`
(`p055`, `p065`), sobre `validation_2024`.


In [ ]:
# [C10] Tabla por semilla y análisis
RUN_ANALYSIS = False
RUN_SELECTION = False
ANALYSIS_CANDIDATE = None   # solo en sensitivity: 'p055' o 'p065'
if RUN_ANALYSIS:
    candidate = ['--candidate', ANALYSIS_CANDIDATE] if ANALYSIS_CANDIDATE else []
    analysis = cs.run_cli([str(PYTHON), '-m', 'uc3m.application.pipeline', 'analyze', '--manifest', str(MANIFEST), '--stage', EVAL_STAGE,
                                      '--split', EVAL_SPLIT] + candidate, cwd=PROJECT_ROOT).report
    print(json.dumps(analysis, indent=1)[:4000])
    if not ANALYSIS_CANDIDATE:
        statistics = cs.run_cli([str(PYTHON), 'tools/eval/build_campaign_statistical_report.py', '--manifest', str(MANIFEST), '--stage', EVAL_STAGE,
                                 '--split', EVAL_SPLIT], cwd=PROJECT_ROOT).report
        print(json.dumps(statistics, indent=1)[:4000])
if RUN_SELECTION:
    selection = cs.run_cli([str(PYTHON), '-m', 'uc3m.application.pipeline', 'select', '--manifest', str(MANIFEST), '--stage', EVAL_STAGE,
                                      '--split', EVAL_SPLIT], cwd=PROJECT_ROOT).report
    print(json.dumps(selection, indent=1)[:4000])
if RUN_ANALYSIS or RUN_SELECTION:
    cs.run_cli(SYNC + ['--sync'], cwd=PROJECT_ROOT, on_line=lambda line: None)


In [ ]:
# [C11] Entrega reproducible
RUN_DELIVERY = False
if RUN_DELIVERY:
    delivery = cs.run_cli([str(PYTHON), '-m', 'uc3m.application.pipeline', 'deliver', '--manifest', str(MANIFEST), '--stage', EVAL_STAGE,
                                      '--split', EVAL_SPLIT, '--destination', str(DRIVE_ROOT.parent/'delivery')],
                          cwd=PROJECT_ROOT).report
    print(json.dumps(delivery, indent=1)[:4000])


## Alcance de validación y fuentes

El cuaderno es cliente de `uc3m.application.training.cli` y de `uc3m.application.pipeline`.
El dataset, los splits y el normalizador vienen congelados en el paquete y se verifican
por hash. Un piloto no sustituye la campaña confirmatoria ni la selección científica.

Sustento: Charbonnier (2023), tesis doctoral, University of Oxford,
https://ora.ox.ac.uk/objects/uuid%3A4acf19ee-76b3-4e71-a424-259cd9e2d6f5;
Costea (2026), tesis de maestría, https://webthesis.biblio.polito.it/40396/;
Pineau et al. (2021), JMLR, https://www.jmlr.org/papers/v22/20-303.html .
Recursos Colab: https://research.google.com/colaboratory/faq.html .
